# 11 · Tensor factorizations / Factorizaciones tensoriales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/11-tensor-factorizations.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#16a34a,rgba(22,163,74,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#16a34a">PART IV · EXERCISE · 15 MIN</span>

Section 10 introduced **Tucker/HOSVD** on the real New York taxi tensor. This section answers the next question:

> **Which tensor decomposition should I use, and what does it cost me?**

We compare **CP, Tucker/HOSVD, Tensor Train and t-SVD** on workshop data, at a matched parameter budget rather than rank for rank.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 .7em">La sección 10 presentó <b>Tucker/HOSVD</b> sobre el tensor real de taxis de Nueva York. Esta sección responde la siguiente pregunta:</div><div style="margin:0 0 .7em"><b>¿Qué descomposición tensorial debo usar y cuánto me cuesta?</b></div><div style="margin:0 0 0">Comparamos <b>CP, Tucker/HOSVD, Tensor Train y t-SVD</b> con datos del taller, usando un presupuesto de parámetros equivalente en lugar de comparar rango contra rango.</div></div>

## What you will be able to do / Lo que podrás hacer

- Explain why flattening can hide the meaning carried by tensor modes.
- Choose between **CP, Tucker/HOSVD, Tensor Train and t-SVD** from the structure of a problem.
- Compare the decompositions by stored parameters, reconstruction error and measured computational cost.
- Compare CP and Tucker at a **matched parameter budget** rather than rank-for-rank.
- Explain why Tensor Train is useful for very high-order tensors.
- Distinguish an exact factorization from truncation and compression.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0">Explicar por qué aplanar un tensor puede ocultar el significado de sus modos.</li><li style="margin:.35em 0">Elegir entre <b>CP, Tucker/HOSVD, Tensor Train y t-SVD</b> según la estructura del problema.</li><li style="margin:.35em 0">Comparar las descomposiciones por parámetros almacenados, error de reconstrucción y coste computacional medido.</li><li style="margin:.35em 0">Comparar CP y Tucker con un <b>presupuesto de parámetros equivalente</b>, en lugar de comparar rango contra rango.</li><li style="margin:.35em 0">Explicar por qué Tensor Train es útil para tensores de orden muy alto.</li><li style="margin:.35em 0">Distinguir una factorización exacta de la truncación y la compresión.</li></ul></div>

## Start here: four models, four mental pictures / Empieza aquí: cuatro modelos, cuatro imágenes mentales

Before touching formulas, keep these four pictures in mind:

| Method / Método | Mental picture / Imagen mental | Best question / Pregunta principal | Main price / Precio principal |
|---|---|---|---|
| **CP** | A tensor is a **sum of simple pieces** / suma de piezas simples | “Can I identify individual components?” | One global rank `R` can be restrictive |
| **Tucker / HOSVD** | Give **each axis its own small basis** / una base pequeña por eje | “How much structure do I keep along each mode?” | The core can grow quickly with order |
| **Tensor Train (TT)** | Pass information through a **chain of small cores** / cadena de núcleos | “Can I store a very high-order tensor?” | Interpretation is more local than CP |
| **t-SVD** | FFT along mode 3, then do **ordinary matrix SVDs** | “Can I use matrix-SVD ideas without flattening mode 3 away?” | This construction is specifically for order-3 tensors |

### The one question for the whole notebook / La pregunta de todo el cuaderno

> **Which structure does my data have, and which decomposition buys that structure at an acceptable cost?**

> 🇪🇸 **¿Qué estructura tienen mis datos y qué descomposición representa esa estructura con un coste aceptable?**

You do **not** need to memorize four algorithms. You need to learn to recognize four situations.

## Step 0 — install the one extra library / Paso 0 — instala la única librería extra

TensorLy gives us reliable implementations of CP, Tucker and Tensor Train.

Run this cell once in a fresh Colab runtime.

> 🇪🇸 TensorLy nos evita programar los algoritmos desde cero. En este notebook queremos **comparar ideas y costes**, no reimplementar optimizadores.

In [ ]:
%pip install -q tensorly

The second dataset option is the **same pinned storm video already used by the workshop**.

`imageio[ffmpeg]` is only needed to decode that verified clip.

> 🇪🇸 La segunda opción de datos es el mismo video de tormenta ya usado en el taller. No estamos añadiendo un dataset nuevo.

In [ ]:
%pip install -q "imageio[ffmpeg]"

## Step 1 — load two tensors we already know / Paso 1 — carga dos tensores que ya conocemos

We will reuse:

1. **NYC taxi tensor**  
   `pickup borough × dropoff borough × hour`

2. **Verified storm clip**  
   `row × column × time`

Every later comparison uses one of these two tensors.

> 🇪🇸 Reutilizaremos el tensor real de taxis de Nueva York y el clip de tormenta verificado. Toda comparación posterior usa uno de los dos.


In [ ]:
import hashlib
import io
import time
import urllib.request
import itertools

import imageio.v3 as iio
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorly as tl

from IPython.display import clear_output, display
from tensorly.decomposition import parafac, tucker, tensor_train

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

tl.set_backend("numpy")
rng = np.random.default_rng(7)

TAXIS = (
    "https://raw.githubusercontent.com/mwaskom/"
    "seaborn-data/master/taxis.csv"
)

STORM_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/1/1e/"
    "Tormenta_en_l%27Almadrava.webm"
)
STORM_SHA256 = "e377fcdd2c79b55bce13c2c24b5dd7e412af39cd400eec548a79d0e59d79dc1b"
UA = "tensors-workshop/1.0 (https://github.com/project-delphi/tensors-workshop)"

def relative_error(a, b):
    return float(np.linalg.norm(np.asarray(a) - np.asarray(b)) / np.linalg.norm(a))

def cp_params(shape, rank):
    # Scalar CP weights can be absorbed into one factor.
    return int(rank * sum(shape))

def tucker_params(shape, ranks):
    return int(np.prod(ranks) + sum(i * r for i, r in zip(shape, ranks)))

def tt_params(shape, internal_rank):
    if len(shape) == 1:
        return int(shape[0])
    ranks = [1] + [int(internal_rank)] * (len(shape) - 1) + [1]
    return int(sum(ranks[k] * shape[k] * ranks[k + 1] for k in range(len(shape))))

def tt_matrix_params(ms, ns, rank):
    total = 0
    for k, (m, n) in enumerate(zip(ms, ns)):
        r_left = 1 if k == 0 else rank
        r_right = 1 if k == len(ms) - 1 else rank
        total += r_left * m * n * r_right
    return int(total)

def best_time(call, repeats=3):
    """Fastest of `repeats` runs — the least noisy estimator, as notebook 12."""
    times, result = [], None
    for _ in range(repeats):
        start = time.perf_counter()
        result = call()
        times.append(time.perf_counter() - start)
    return min(times), result

def tsvd_reconstruct(A, tubal_rank=None):
    """FFT on mode 3, matrix SVD per frontal slice, inverse FFT."""
    A = np.asarray(A, dtype=float)
    if A.ndim != 3:
        raise ValueError("t-SVD here is defined for an order-3 tensor.")
    Af = np.fft.fft(A, axis=2)
    Rf = np.empty_like(Af, dtype=complex)
    for k in range(A.shape[2]):
        U, s, Vh = np.linalg.svd(Af[:, :, k], full_matrices=False)
        keep = len(s) if tubal_rank is None else min(int(tubal_rank), len(s))
        Rf[:, :, k] = (U[:, :keep] * s[:keep]) @ Vh[:keep]
    return np.fft.ifft(Rf, axis=2).real

def fit_tucker_error(A, ranks):
    model = tucker(tl.tensor(A), rank=list(ranks), init="svd",
                   n_iter_max=80, tol=1e-7)
    return relative_error(A, tl.tucker_to_tensor(model))

def best_tucker_ranks_for_budget(A, budget, max_rank=8, tol=0.10, keep=10):
    """Tucker ranks that spend `budget` parameters as well as possible.

    Two traps this avoids. Picking the candidate whose parameter count is
    merely CLOSEST to the budget optimises the wrong quantity — a tuple can
    land one parameter away by collapsing a whole mode to rank 1, which is a
    worse model than a tuple sitting further from the budget. And it can land
    there with FEWER parameters than an alternative, so the comparison is not
    even generous to Tucker. So: shortlist the tuples within `tol` of the
    budget (nearest ones if none qualify), fit each, and keep the lowest
    error, breaking ties toward fewer parameters.
    """
    A = np.asarray(A)
    shape = A.shape
    limits = [range(1, min(int(i), max_rank) + 1) for i in shape]
    scored = []
    for ranks in itertools.product(*limits):
        p = tucker_params(shape, ranks)
        scored.append((abs(p - budget), p, tuple(int(r) for r in ranks)))
    scored.sort()
    near = [c for c in scored if c[0] <= tol * budget][:keep] or scored[:keep]
    return min((fit_tucker_error(A, ranks), p, ranks)
               for _, p, ranks in near)[2]

# --- Real NYC taxi tensor: same construction as notebook 10 ---
taxis = pd.read_csv(TAXIS)
taxis["pickup_dt"] = pd.to_datetime(taxis["pickup"], errors="coerce")
taxis["hour"] = taxis["pickup_dt"].dt.hour
sub = taxis.dropna(subset=["pickup_borough", "dropoff_borough", "hour"]).copy()
sub["hour"] = sub["hour"].astype(int)

pickup_names = sorted(sub["pickup_borough"].unique())
dropoff_names = sorted(sub["dropoff_borough"].unique())
pickup_index = {name: i for i, name in enumerate(pickup_names)}
dropoff_index = {name: i for i, name in enumerate(dropoff_names)}

T_taxi = np.zeros((len(pickup_names), len(dropoff_names), 24), dtype=float)
for (p, d, h), count in sub.groupby(
    ["pickup_borough", "dropoff_borough", "hour"]
).size().items():
    T_taxi[pickup_index[p], dropoff_index[d], int(h)] = float(count)

# --- Same pinned storm clip as notebooks 02/05, kept small after verification ---
def fetch_verified_storm(n_frames=10, stride=45, crop=32):
    req = urllib.request.Request(STORM_URL, headers={"User-Agent": UA})
    raw = urllib.request.urlopen(req, timeout=120).read()
    got = hashlib.sha256(raw).hexdigest()
    if got != STORM_SHA256:
        raise ValueError(f"checksum mismatch: expected {STORM_SHA256}, got {got}")

    kept = []
    for frame_i, frame in enumerate(
        iio.imiter(io.BytesIO(raw), plugin="FFMPEG", extension=".webm")
    ):
        if frame_i % stride == 0:
            gray = frame[..., :3].mean(axis=2) / 255.0
            r0 = gray.shape[0] // 2 - crop // 2
            c0 = gray.shape[1] // 2 - crop // 2
            kept.append(gray[r0:r0 + crop, c0:c0 + crop])
            if len(kept) == n_frames:
                break

    # spatial row × spatial column × time
    return np.moveaxis(np.stack(kept), 0, 2)

T_storm = fetch_verified_storm()

print("Taxi tensor / Tensor taxis:", T_taxi.shape, "entries:", T_taxi.size)
print("Storm tensor / Tensor tormenta:", T_storm.shape, "entries:", T_storm.size)
print("Storm SHA-256 verified / verificado:", STORM_SHA256[:12] + "…")
print("EN: setup ready.")
print("ES: preparación lista.")

### Two words before we start / Dos palabras antes de empezar

A fair comparison matches parameter budget, not rank. Different models spend rank differently.

**Parameter budget:** how many numbers you may store.

**Rank:** CP rank, Tucker mode ranks, TT bond dimensions, or tubal rank. Each controls a different structure.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Presupuesto: cuántos números puedes guardar. El rango cambia según el modelo: CP, rangos por modo de Tucker, dimensiones de enlace de TT o rango tubular.</div>

In [ ]:
toy = np.arange(24).reshape(2, 3, 4)
print("toy.shape =", toy.shape)
print("toy.ndim  =", toy.ndim)
print("toy[1, 2, 3] =", toy[1, 2, 3])
print()
print("Axes / Ejes:")
print("  0 → patient / paciente")
print("  1 → marker / marcador")
print("  2 → visit / visita")


## Step 2 — Why can flattening be a problem? / Paso 2 — ¿Por qué aplanar puede ser un problema?

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#16a34a,rgba(22,163,74,0))"></div>

Flattening can hide which axes belong together. Test the representation against the structure you need.

A synthetic mixture has shape `(20,24,18)`: `sample × emission × excitation`.

Estimate three dye amounts per sample. CP keeps factors for each axis. Flatten + SVD groups emission and excitation.

Reshaping preserves every value. The models impose different structure. This example is not a universal ranking.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La mezcla sintética tiene forma `(20,24,18)`: muestra × emisión × excitación. Estima tres tintes por muestra. CP conserva factores por eje; SVD agrupa dos ejes. Aplanar conserva los valores. Este ejemplo no establece un ganador universal.</div>

In [ ]:
# Synthetic by design, reproduced from Ravi's blog generator (seed 7).
MIX_SHAPE = (20, 24, 18)
MIX_RANK = 3
NOISE_FRAC = 0.08

def bump(n, center, width):
    grid = np.linspace(0.0, 1.0, n)
    return np.exp(-0.5 * ((grid - center) / width) ** 2)

def make_mixing_cube(seed=7):
    local = np.random.default_rng(seed)
    ns, ne, nx = MIX_SHAPE
    sample_centres, sample_width = (0.32, 0.50, 0.68), 0.20
    em_centres, em_width = (0.18, 0.50, 0.82), 0.09
    ex_centres, ex_width = (0.20, 0.52, 0.84), 0.09

    factors = [
        np.zeros((ns, MIX_RANK)),
        np.zeros((ne, MIX_RANK)),
        np.zeros((nx, MIX_RANK)),
    ]
    terms = []
    for r in range(MIX_RANK):
        a = bump(ns, sample_centres[r], sample_width)
        b = bump(ne, em_centres[r], em_width)
        c = bump(nx, ex_centres[r], ex_width)
        factors[0][:, r], factors[1][:, r], factors[2][:, r] = a, b, c
        terms.append(a[:, None, None] * b[None, :, None] * c[None, None, :])

    clean = sum(terms)
    noisy = clean + NOISE_FRAC * clean.std() * local.normal(size=clean.shape)
    return noisy, factors

def best_column_alignment(true, estimate):
    t = true / np.linalg.norm(true, axis=0, keepdims=True)
    e = estimate / np.linalg.norm(estimate, axis=0, keepdims=True)
    C = t.T @ e
    best = max(itertools.permutations(range(C.shape[1])),
               key=lambda p: sum(abs(C[i, p[i]]) for i in range(C.shape[0])))
    aligned = estimate[:, best].copy()
    corr = []
    for i in range(true.shape[1]):
        sign = np.sign(np.dot(true[:, i], aligned[:, i])) or 1.0
        aligned[:, i] *= sign
        corr.append(abs(np.dot(
            true[:, i] / np.linalg.norm(true[:, i]),
            aligned[:, i] / np.linalg.norm(aligned[:, i])
        )))
    return aligned, np.array(corr)

mix_cube, true_mix_factors = make_mixing_cube()

cp_mix = parafac(
    tl.tensor(mix_cube), rank=3, init="svd", random_state=7,
    n_iter_max=200, tol=1e-8
)
cp_mix_recon = tl.cp_to_tensor(cp_mix)
cp_sample, cp_sample_corr = best_column_alignment(
    true_mix_factors[0], np.asarray(cp_mix.factors[0])
)

flat = mix_cube.reshape(mix_cube.shape[0], -1)
U_flat, _, _ = np.linalg.svd(flat, full_matrices=False)
svd_sample, svd_sample_corr = best_column_alignment(
    true_mix_factors[0], U_flat[:, :3]
)

print("Synthetic cube / Cubo sintético:", mix_cube.shape)
print("CP relative error / Error relativo CP:", f"{relative_error(mix_cube, cp_mix_recon):.3f}")
print("CP mean amount correlation / Correlación CP:", f"{cp_sample_corr.mean():.3f}")
print("Flatten+SVD mean |corr| / Aplanar+SVD:", f"{svd_sample_corr.mean():.3f}")
print("Flatten+SVD negative amount entries / Entradas negativas:",
      int((svd_sample < 0).sum()))

### Reading the plot / Cómo leer la gráfica

Pick one dye.

Compare the true pattern with CP and SVD across 20 samples.

The curves are normalized. Compare their shapes, not absolute dye amounts.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Compara el patrón verdadero con CP y SVD en 20 muestras. Las curvas están normalizadas: compara formas, no cantidades absolutas.</div>

In [ ]:
#@title Dye patterns / Patrones de tintes — run me / ejecútame { display-mode: 'form' }

# Normalize each column only for visualization.
# This removes arbitrary factor scaling without changing correlation.
def normalize_pattern(v):
    v = np.asarray(v, dtype=float)
    scale = np.max(np.abs(v))
    return v if scale == 0 else v / scale

true_patterns = np.column_stack([
    normalize_pattern(true_mix_factors[0][:, j])
    for j in range(3)
])
cp_patterns = np.column_stack([
    normalize_pattern(cp_sample[:, j])
    for j in range(3)
])
svd_patterns = np.column_stack([
    normalize_pattern(svd_sample[:, j])
    for j in range(3)
])

dye_choice = widgets.ToggleButtons(
    options=[
        ("Dye 1 / Tinte 1", 0),
        ("Dye 2 / Tinte 2", 1),
        ("Dye 3 / Tinte 3", 2),
    ],
    value=0,
    description="Dye / Tinte:",
    style={"description_width": "85px"},
)

def show_dye(dye):
    dye = int(dye)
    samples = np.arange(1, MIX_SHAPE[0] + 1)

    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    ax.plot(samples, true_patterns[:, dye], marker="o", label="Truth / Verdad")
    ax.plot(samples, cp_patterns[:, dye], marker="o", label="CP")
    ax.plot(
        samples,
        svd_patterns[:, dye],
        marker="o",
        label="Flatten + SVD / Aplanar + SVD",
    )
    ax.axhline(0, linewidth=0.8)
    ax.set_xlabel("Sample / Muestra")
    ax.set_ylabel("Relative amount pattern / Patrón relativo")
    ax.set_title(
        f"Dye {dye + 1}: which method follows the truth?\n"
        f"Tinte {dye + 1}: ¿qué método sigue mejor la verdad?"
    )
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

    cp_corr = float(cp_sample_corr[dye])
    svd_corr = float(svd_sample_corr[dye])

    print("What does this graph mean? / ¿Qué significa esta gráfica?")
    print()
    print(f"CP correlation / Correlación CP: {cp_corr:.3f}")
    print(f"Flatten + SVD correlation / Correlación Aplanar + SVD: {svd_corr:.3f}")
    print()

    if cp_corr > svd_corr:
        print("✅ EN: CP follows the true dye pattern more closely.")
        print("✅ ES: CP sigue más de cerca el patrón verdadero del tinte.")
    else:
        print("EN: In this selected component, SVD happens to be closer.")
        print("ES: En este componente seleccionado, SVD resulta más cercano.")

    if np.any(svd_patterns[:, dye] < 0):
        print()
        print("⚠️ EN: Flatten + SVD goes below zero for a nonnegative physical amount.")
        print("⚠️ ES: Aplanar + SVD baja de cero para una cantidad física no negativa.")

fluorescence_output = widgets.interactive_output(
    show_dye,
    {"dye": dye_choice},
)

display(widgets.VBox([
    widgets.HTML(
        "<b>Choose one dye and compare the three lines. / "
        "Elige un tinte y compara las tres líneas.</b>"
    ),
    dye_choice,
    fluorescence_output,
]))

## Step 3 — Four tensor decompositions / Paso 3 — Cuatro descomposiciones tensoriales

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#16a34a,rgba(22,163,74,0))"></div>

CP: simple additive parts. Tucker: a small core plus mode maps. TT: a chain. t-SVD: frequency-wise matrices.

| Model / Modelo | Useful structure / Estructura útil |
|---|---|
| CP | One pattern per axis per component / Un patrón por eje y componente |
| Tucker | Core plus a separate rank per axis / Núcleo y rango propio por eje |
| TT | Chain of cores for many axes / Cadena de núcleos para muchos ejes |
| t-SVD | FFT → matrix SVDs → inverse FFT / FFT → SVD matriciales → FFT inversa |

For t-SVD, FFT runs along the third axis: NumPy `axis=2`.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Relaciona cada modelo con la estructura que necesitas. En t-SVD, la FFT recorre el tercer eje: `axis=2`.</div>

### Interactive method chooser / Selector interactivo

Start from a **situation**, not from a method name.

Then inspect which assumptions caused the recommendation.

The controls below deliberately ask every question that changes the answer:

- tensor order;
- whether modes carry distinct meanings;
- uniqueness;
- exact t-SVD algebra;
- nonnegativity;
- very high order;
- mode-specific ranks.

In [ ]:
#@title Method chooser / Selector de método — run me / ejecútame { display-mode: 'form' }

scenario_w = widgets.Dropdown(
    options=[
        ("Choose manually / Elegir manualmente", "manual"),
        ("Separate physical sources / Separar fuentes físicas", "sources"),
        ("Compress pickup × dropoff × hour / Comprimir taxis", "taxi"),
        ("Very high-order tensor / Tensor de orden muy alto", "high"),
        ("Video-like order-3 tensor / Tensor tipo video", "video"),
    ],
    value="manual",
    description="Situation:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="650px"),
)

order_w = widgets.IntSlider(
    value=3, min=3, max=10, step=1,
    description="Order / Orden:",
    continuous_update=False,
    style={"description_width": "130px"},
)
meaning_w = widgets.ToggleButtons(
    options=[
        ("Modes mean different things / Modos distintos", True),
        ("Modes are mostly interchangeable / Similares", False),
    ],
    value=True,
    description="Axes:",
    style={"description_width": "70px"},
)
unique_w = widgets.Checkbox(value=False, description="Need unique components / Componentes únicos")
exact_w = widgets.Checkbox(value=False, description="Need exact t-SVD algebra / Álgebra t-SVD")
nonneg_w = widgets.Checkbox(value=False, description="Nonnegative quantities / Cantidades ≥ 0")
mode_rank_w = widgets.Checkbox(value=True, description="Different rank per mode / Rango por modo")
high_order_w = widgets.Checkbox(value=False, description="Very high order / Orden muy alto")
chooser_out = widgets.Output()

PRESETS = {
    "sources": dict(order=3, meaning=True, unique=True, exact=False, nonneg=True, mode_rank=False, high=False),
    "taxi": dict(order=3, meaning=True, unique=False, exact=False, nonneg=True, mode_rank=True, high=False),
    "high": dict(order=8, meaning=True, unique=False, exact=False, nonneg=False, mode_rank=False, high=True),
    "video": dict(order=3, meaning=True, unique=False, exact=True, nonneg=False, mode_rank=False, high=False),
}

def apply_scenario(change):
    key = change["new"]
    if key == "manual":
        explain_choice()
        return
    p = PRESETS[key]
    order_w.value = p["order"]
    meaning_w.value = p["meaning"]
    unique_w.value = p["unique"]
    exact_w.value = p["exact"]
    nonneg_w.value = p["nonneg"]
    mode_rank_w.value = p["mode_rank"]
    high_order_w.value = p["high"]
    explain_choice()

def explain_choice(*_):
    order = int(order_w.value)
    with chooser_out:
        clear_output(wait=True)

        if exact_w.value:
            if order != 3:
                print("⚠️ t-SVD in this notebook is defined for order 3.")
                print("   En este notebook, t-SVD se usa para tensores de orden 3.")
                return
            method = "t-SVD"
            why = "mode 3 is preserved through an FFT instead of being flattened away"
            remember = "FFT → matrix SVDs → inverse FFT"
        elif high_order_w.value or order >= 5:
            method = "Tensor Train (TT)"
            why = "a chain of small cores avoids a Tucker core that grows like r^N"
            remember = "high order → think chain"
        elif unique_w.value:
            method = "CP"
            why = "rank-1 components can be essentially unique under Kruskal-type conditions"
            if nonneg_w.value:
                why += "; nonnegativity also matches the physical quantity"
            remember = "named pieces → CP"
        elif mode_rank_w.value and meaning_w.value:
            method = "Tucker / HOSVD"
            why = "each semantic mode can keep a different number of directions"
            remember = "one small basis per axis"
        elif nonneg_w.value:
            method = "CP with nonnegative constraints"
            why = "additive components respect quantities that cannot be negative"
            remember = "positive parts → CP/NCP"
        else:
            method = "Tucker / HOSVD"
            why = "it is the flexible low-order choice when mode-specific subspaces matter"
            remember = "flexible order-3 compression"

        print("✅ Recommended / Recomendado:", method)
        print("🧭 Because / Porque:", why)
        print("🧠 Memory hook / Regla para recordar:", remember)

scenario_w.observe(apply_scenario, names="value")
for w in [order_w, meaning_w, unique_w, exact_w, nonneg_w, mode_rank_w, high_order_w]:
    w.observe(explain_choice, names="value")

display(widgets.VBox([
    widgets.HTML("<b>1. Pick a situation. 2. Inspect the assumptions. 3. Change one assumption and see the method change.</b>"),
    scenario_w,
    order_w,
    meaning_w,
    widgets.HBox([unique_w, exact_w]),
    widgets.HBox([nonneg_w, mode_rank_w]),
    high_order_w,
    chooser_out,
]))
explain_choice()

## Step 4 — what does each representation store? / Paso 4 — ¿qué almacena cada representación?

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#16a34a,rgba(22,163,74,0))"></div>

Count stored numbers before comparing error. Compression is a budget decision.

Let order be `N`. Every mode has size `I`.

| Model / Modelo | Stored values / Valores guardados |
|---|---|
| Dense / Denso | `I^N` |
| CP, rank / rango `R` | `RNI` (weights absorbed / pesos absorbidos) |
| Tucker, mode ranks / rangos `r` | `r^N + NIr` |
| TT, internal ranks / rangos internos `r`, `N ≥ 2` | `2Ir + (N−2)Ir²` |

Hold sizes and ranks fixed. Predict which curves grow exponentially. Keeping a fixed error may require ranks to grow.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Fija tamaños y rangos. Predice qué curvas crecen exponencialmente. TT crece linealmente con el orden a rango fijo. Mantener un error dado puede exigir rangos mayores.</div>

In [ ]:
orders = np.arange(3, 11)
I_demo, R_demo, r_demo = 12, 4, 4

dense_curve = np.array([I_demo ** N for N in orders], dtype=float)
cp_curve = np.array([R_demo * N * I_demo for N in orders], dtype=float)
tucker_curve = np.array([r_demo ** N + N * I_demo * r_demo for N in orders], dtype=float)
tt_curve = np.array([
    2 * I_demo * r_demo + max(N - 2, 0) * I_demo * r_demo ** 2
    for N in orders
], dtype=float)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
for label, curve in [
    ("Dense", dense_curve),
    ("CP (fixed global R)", cp_curve),
    ("Tucker", tucker_curve),
    ("TT", tt_curve),
]:
    ax.plot(orders, curve, marker="o", label=label)
ax.set_yscale("log")
ax.set_xlabel("tensor order N / orden N")
ax.set_ylabel("stored numbers / números almacenados")
ax.set_title("The curse is a storage curve, not a slogan")
ax.grid(alpha=0.25, which="both")
ax.legend()
plt.show()

print("At N=10 / En N=10:")
for name, curve in [("Dense", dense_curve), ("CP", cp_curve),
                    ("Tucker", tucker_curve), ("TT", tt_curve)]:
    print(f"  {name:8s}: {int(curve[-1]):,}")

### Now measure time / Ahora mide tiempo

Big-O predicts direction, not one exact time. Time several sizes and look for the trend.

Time four models on small storm-video crops.

Fit `log(time) ≈ p log(entries) + constant`. This slope describes these crops and algorithm settings, not a universal complexity exponent.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Mide los cuatro modelos con recortes del video. Ajusta `log(tiempo)` frente a `log(entradas)`. La pendiente describe estos recortes y parámetros, no una complejidad universal.</div>

In [ ]:
crop_sizes = np.array([12, 16, 24, 32])
timing_tensors = {
    int(s): T_storm[
        T_storm.shape[0] // 2 - s // 2:T_storm.shape[0] // 2 + s // 2,
        T_storm.shape[1] // 2 - s // 2:T_storm.shape[1] // 2 + s // 2,
        :8,
    ]
    for s in crop_sizes
}

def timed_cp(A):
    return parafac(tl.tensor(A), rank=3, init="svd", random_state=7,
                   n_iter_max=35, tol=1e-6)

def timed_tucker(A):
    ranks = tuple(min(3, d) for d in A.shape)
    return tucker(tl.tensor(A), rank=ranks, init="svd",
                  n_iter_max=35, tol=1e-6)

def timed_tt(A):
    return tensor_train(tl.tensor(A), rank=3)

def timed_tsvd(A):
    return tsvd_reconstruct(A, tubal_rank=min(3, A.shape[0], A.shape[1]))

TIMED = {
    "CP": timed_cp,
    "Tucker": timed_tucker,
    "TT": timed_tt,
    "t-SVD": timed_tsvd,
}

timing_rows = []
for name, fn in TIMED.items():
    times = []
    entries = []
    for s in crop_sizes:
        A = timing_tensors[int(s)]
        elapsed, _ = best_time(lambda A=A, fn=fn: fn(A))
        times.append(elapsed)
        entries.append(A.size)
    slope = np.polyfit(np.log(entries), np.log(times), 1)[0]
    timing_rows.append((name, slope, times))

print("Empirical log(time) ~ p log(entries) / Pendiente empírica")
for name, slope, times in timing_rows:
    print(f"{name:8s}: p={slope:5.2f}   "
          + " ".join(f"{1000*t:7.1f} ms" for t in times))

fig, ax = plt.subplots(figsize=(8.5, 4.4))
for name, slope, times in timing_rows:
    ax.loglog([timing_tensors[int(s)].size for s in crop_sizes],
              times, "o-", label=f"{name} (p={slope:.2f})")
ax.set_xlabel("number of real tensor entries / entradas reales")
ax.set_ylabel("seconds / segundos")
ax.set_title("Measured scaling on verified storm pixels")
ax.grid(alpha=0.25, which="both")
ax.legend()
plt.show()

## Step 5 — CP vs Tucker: a fair comparison / Paso 5 — CP vs Tucker: una comparación justa

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#16a34a,rgba(22,163,74,0))"></div>

Hold the parameter budget fixed. Then ask which structure each model preserves.

Choose taxi data or storm video. Move the CP rank.

The notebook selects Tucker ranks with a similar parameter count. Compare actual counts and relative error.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige taxis o video y cambia el rango CP. El notebook busca rangos Tucker con un número de parámetros similar. Compara los conteos reales y el error relativo.</div>

In [ ]:
DATASETS = {
    "taxi": T_taxi,
    "storm": T_storm,
}
CP_RANKS = range(1, 7)
tradeoff_cache = {}

for data_name, A in DATASETS.items():
    shape = A.shape
    rows = []
    for R in CP_RANKS:
        budget = cp_params(shape, R)
        tranks = best_tucker_ranks_for_budget(A, budget, max_rank=6)

        cp_model = parafac(
            tl.tensor(A), rank=R, init="svd", random_state=7,
            n_iter_max=120, tol=1e-7
        )
        cp_rec = tl.cp_to_tensor(cp_model)

        tuck_model = tucker(
            tl.tensor(A), rank=tranks, init="svd",
            n_iter_max=80, tol=1e-7
        )
        tuck_rec = tl.tucker_to_tensor(tuck_model)

        rows.append({
            "cp_rank": R,
            "cp_params": cp_params(shape, R),
            "cp_error": relative_error(A, cp_rec),
            "tucker_ranks": tranks,
            "tucker_params": tucker_params(shape, tranks),
            "tucker_error": relative_error(A, tuck_rec),
            "cp_model": cp_model,
        })
    tradeoff_cache[data_name] = rows

print("Matched-budget curves ready / Curvas de presupuesto igualado listas.")

In [ ]:
#@title CP and Tucker / CP y Tucker — run me / ejecútame { display-mode: 'form' }

dataset_w = widgets.Dropdown(
    options=[
        (
            "NYC taxi: pickup × dropoff × hour / "
            "Taxis NYC: origen × destino × hora",
            "taxi",
        ),
        (
            "Verified storm video / Video verificado de tormenta",
            "storm",
        ),
    ],
    value="taxi",
    description="Example / Ejemplo:",
    style={"description_width": "125px"},
    layout=widgets.Layout(width="720px"),
)

rank_w = widgets.IntSlider(
    value=3,
    min=1,
    max=6,
    step=1,
    description="CP rank / Rango CP:",
    continuous_update=False,
    style={"description_width": "135px"},
    layout=widgets.Layout(width="560px"),
)

def show_tradeoff_simple(data_name, R):
    R = int(R)
    row = tradeoff_cache[data_name][R - 1]

    cp_params_now = int(row["cp_params"])
    tu_params_now = int(row["tucker_params"])
    cp_err = float(row["cp_error"])
    tu_err = float(row["tucker_error"])

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))

    axes[0].bar(
        ["CP", "Tucker"],
        [cp_params_now, tu_params_now],
    )
    axes[0].set_ylabel("Stored numbers / Números guardados")
    axes[0].set_title("1. Similar storage budget\n1. Presupuesto parecido")
    axes[0].grid(axis="y", alpha=0.2)

    axes[1].bar(
        ["CP", "Tucker"],
        [cp_err, tu_err],
    )
    axes[1].set_ylabel("Reconstruction error / Error")
    axes[1].set_title("2. Which reconstruction is closer?\n2. ¿Cuál reconstruye mejor?")
    axes[1].grid(axis="y", alpha=0.2)

    plt.tight_layout()
    plt.show()

    print("What are we comparing? / ¿Qué estamos comparando?")
    print()
    print(f"CP rank / Rango CP: {R}")
    print(f"CP parameters / Parámetros CP: {cp_params_now:,}")
    print(
        "Tucker ranks / Rangos Tucker:",
        row["tucker_ranks"],
    )
    print(f"Tucker parameters / Parámetros Tucker: {tu_params_now:,}")
    print()
    print(f"CP reconstruction error / Error CP: {cp_err:.4f}")
    print(f"Tucker reconstruction error / Error Tucker: {tu_err:.4f}")
    print()

    if abs(cp_err - tu_err) < 0.01:
        print("EN: Their reconstruction errors are very similar at this storage budget.")
        print("ES: Sus errores de reconstrucción son muy parecidos con este presupuesto.")
    elif cp_err < tu_err:
        print("✅ EN: CP reconstructs the tensor more closely at this similar storage budget.")
        print("✅ ES: CP reconstruye mejor el tensor con un almacenamiento parecido.")
    else:
        print("✅ EN: Tucker reconstructs the tensor more closely at this similar storage budget.")
        print("✅ ES: Tucker reconstruye mejor el tensor con un almacenamiento parecido.")

    print()
    print("Important / Importante:")
    print("EN: Lower error does not automatically mean 'better for every purpose'.")
    print("ES: Menor error no significa automáticamente 'mejor para cualquier objetivo'.")

trade_output = widgets.interactive_output(
    show_tradeoff_simple,
    {"data_name": dataset_w, "R": rank_w},
)

display(widgets.VBox([
    widgets.HTML(
        "<b>1. Choose an example. 2. Move the rank. 3. Compare the two bar charts."
        "<br>1. Elige un ejemplo. 2. Mueve el rango. 3. Compara las dos gráficas.</b>"
    ),
    dataset_w,
    rank_w,
    trade_output,
]))

## Exercise 1 — same budget, different structure / Ejercicio 1 — mismo presupuesto, distinta estructura

### Goal / Objetivo

Compare **CP and Tucker on the real taxi tensor** without giving one model more parameters.

### Do it in four steps

1. Choose a CP rank `R`.
2. Compute `CP parameters = R(I + J + K)`.
3. Find Tucker ranks whose parameter count is close.
4. Compare:
   - relative reconstruction error;
   - parameter count;
   - interpretability of the factors.

### Before running / Antes de ejecutar

Predict:

> Will the method with lower error necessarily be the method you would choose if a human must interpret the components?

That is the point of the exercise.

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Use CP rank R = 3 on T_taxi.
# 2. Compute its parameter budget R * sum(T_taxi.shape).
# 3. Find Tucker ranks with a parameter count as close as possible.
# 4. Fit both decompositions and reconstruct both tensors.
# 5. Compare relative error and parameter count.
# 6. Inspect the CP hour factors. Why are individual CP components easier
#    to name than Tucker columns plus a dense core?
#
# ES:
# 1. Usa CP de rango R = 3 sobre T_taxi.
# 2. Calcula su presupuesto de parámetros.
# 3. Busca rangos Tucker con un presupuesto lo más parecido posible.
# 4. Ajusta y reconstruye ambos modelos.
# 5. Compara error relativo y parámetros.
# 6. Inspecciona los factores horarios CP e interpreta la diferencia.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

R = 3
shape = T_taxi.shape
budget = cp_params(shape, R)
ranks_t = best_tucker_ranks_for_budget(T_taxi, budget, max_rank=6)

cp_fit = parafac(tl.tensor(T_taxi), rank=R, init="svd",
                 random_state=7, n_iter_max=200, tol=1e-8)
cp_hat = tl.cp_to_tensor(cp_fit)

tu_fit = tucker(tl.tensor(T_taxi), rank=ranks_t, init="svd",
                n_iter_max=120, tol=1e-8)
tu_hat = tl.tucker_to_tensor(tu_fit)

print("Taxi shape:", shape)
print("CP:", cp_params(shape, R), "params",
      "error", f"{relative_error(T_taxi, cp_hat):.4f}")
print("Tucker:", ranks_t, tucker_params(shape, ranks_t), "params",
      "error", f"{relative_error(T_taxi, tu_hat):.4f}")
print()
print("EN: equal storage does not make the models equivalent. CP spends the")
print("    budget on individually inspectable rank-1 components; Tucker spends")
print("    some of it on a core that lets mode components interact.")
print("ES: igual almacenamiento no hace equivalentes a los modelos: CP compra")
print("    componentes individuales; Tucker compra interacciones mediante el núcleo.")

## Step 6 — Tensor compression inside a neural network / Paso 6 — Compresión tensorial dentro de una red neuronal

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#16a34a,rgba(22,163,74,0))"></div>

Tensorize a layer when its axes carry structure you want to keep. Compression is useful only if the task stays good enough.

A `3 × 3 × 512 × 512` kernel holds 2,359,296 weights. CP rank 64 holds `64(3+3+512+512) = 65,920`, with component weights absorbed.

The widget fits **small synthetic layers** and estimates storage for full-size layers. Toy reconstruction error does not measure a trained network's accuracy.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El kernel completo guarda 2.359.296 pesos; CP de rango 64 guarda 65.920 con pesos absorbidos. El widget ajusta capas sintéticas pequeñas y estima almacenamiento a tamaño completo. Su error no mide el acierto de una red entrenada.</div>

In [ ]:
# Local synthetic stand-ins for the two layers described above. They are small
# enough to refit at every slider position; the storage numbers the widget
# prints are still those of the full-size layers.
CONV_TOY_D = 3
CONV_TOY_C = 64
CONV_TRUE_RANK = 16
TOY_NOISE_FRAC = 0.08

def make_cp_kernel(seed=7, d=CONV_TOY_D, channels=CONV_TOY_C,
                   rank=CONV_TRUE_RANK):
    local = np.random.default_rng(seed)
    factors = [
        local.normal(size=(d, rank)),
        local.normal(size=(d, rank)),
        local.normal(size=(channels, rank)),
        local.normal(size=(channels, rank)),
    ]
    clean = tl.cp_to_tensor((np.ones(rank), factors))
    noisy = clean + TOY_NOISE_FRAC * clean.std() * local.normal(size=clean.shape)
    return np.asarray(noisy), np.asarray(clean)

def fit_cp_array(A, rank):
    model = parafac(tl.tensor(A), rank=int(rank), n_iter_max=120,
                    init="svd", random_state=7)
    return np.asarray(tl.cp_to_tensor(model))

def random_tt_cores(seed, ms, ns, rank):
    local = np.random.default_rng(seed)
    cores = []
    for k in range(len(ms)):
        rl = 1 if k == 0 else rank
        rr = 1 if k == len(ms) - 1 else rank
        core = local.normal(size=(rl, ms[k], ns[k], rr))
        cores.append(core / np.sqrt(core.size))
    return cores

def tt_matrix_to_dense(cores):
    acc = cores[0][0]
    for core in cores[1:]:
        acc = np.tensordot(acc, core, axes=([-1], [0]))
        rows, cols, mk, nk, rnext = acc.shape
        acc = np.transpose(acc, (0, 2, 1, 3, 4)).reshape(
            rows * mk, cols * nk, rnext
        )
    return np.asarray(acc[..., 0])

def tt_matrix_svd(matrix, ms, ns, max_rank):
    order = len(ms)
    tensor = matrix.reshape(tuple(ms) + tuple(ns))
    axes = [ax for k in range(order) for ax in (k, order + k)]
    remaining = np.transpose(tensor, axes)
    cores, ranks = [], [1]
    for k in range(order - 1):
        rl = ranks[-1]
        mk, nk = ms[k], ns[k]
        rest = remaining.size // (rl * mk * nk)
        unfolding = remaining.reshape(rl * mk * nk, rest)
        u, s, vt = np.linalg.svd(unfolding, full_matrices=False)
        keep = min(int(max_rank), u.shape[1])
        cores.append(u[:, :keep].reshape(rl, mk, nk, keep))
        remaining = s[:keep, None] * vt[:keep]
        ranks.append(keep)
    cores.append(remaining.reshape(ranks[-1], ms[-1], ns[-1], 1))
    return cores

conv_toy, _ = make_cp_kernel()

TT_TOY_MS = [4, 4, 4, 4]
TT_TOY_NS = [4, 4, 4, 4]
TT_TRUE_RANK = 4
tt_toy_clean = tt_matrix_to_dense(
    random_tt_cores(7, TT_TOY_MS, TT_TOY_NS, rank=TT_TRUE_RANK)
)
tt_local = np.random.default_rng(8)
tt_toy = tt_toy_clean + TOY_NOISE_FRAC * tt_toy_clean.std() * tt_local.normal(
    size=tt_toy_clean.shape
)

conv_error_cache = {}
tt_error_cache = {}

def conv_error_for_rank(rank):
    rank = int(rank)
    if rank not in conv_error_cache:
        conv_error_cache[rank] = relative_error(
            conv_toy, fit_cp_array(conv_toy, rank)
        )
    return conv_error_cache[rank]

def tt_error_for_rank(rank):
    rank = int(rank)
    if rank not in tt_error_cache:
        cores = tt_matrix_svd(tt_toy, TT_TOY_MS, TT_TOY_NS, rank)
        tt_error_cache[rank] = relative_error(
            tt_toy, tt_matrix_to_dense(cores)
        )
    return tt_error_cache[rank]

print("Stand-in conv kernel / Kernel sustituto:", conv_toy.shape,
      "- true CP rank / rango CP verdadero:", CONV_TRUE_RANK)
print("Stand-in TT-matrix / Matriz TT sustituta:", tt_toy.shape,
      "- true TT rank / rango TT verdadero:", TT_TRUE_RANK)
print("EN: error is measured on these; storage is quoted for the full layers.")
print("ES: el error se mide aquí; el almacenamiento es el de las capas completas.")

In [ ]:
#@title Layer compression / Compresión de capas — run me / ejecútame { display-mode: 'form' }

layer_w = widgets.Dropdown(
    options=[
        (
            "Convolution 3×3×512×512 with CP / "
            "Convolución 3×3×512×512 con CP",
            "conv",
        ),
        (
            "Transformer 4096×4096 with TT / "
            "Transformer 4096×4096 con TT",
            "tt",
        ),
    ],
    value="conv",
    description="Example / Ejemplo:",
    style={"description_width": "125px"},
    layout=widgets.Layout(width="720px"),
)

layer_rank = widgets.IntSlider(
    value=16,
    min=1,
    max=64,
    step=1,
    description="Rank / Rango:",
    continuous_update=False,
    style={"description_width": "105px"},
    layout=widgets.Layout(width="560px"),
)

def update_rank_limit(change):
    if change["new"] == "conv":
        layer_rank.max = 64
        if layer_rank.value > 64:
            layer_rank.value = 64
    else:
        layer_rank.max = 16
        if layer_rank.value > 16:
            layer_rank.value = 16

layer_w.observe(update_rank_limit, names="value")

def show_layer_simple(layer, rank):
    rank = int(rank)

    if layer == "conv":
        d = 3
        cin = cout = 512
        H = W = 14

        dense_weights = d * d * cin * cout
        compressed_weights = rank * (2 * d + cin + cout)

        dense_macs = H * W * dense_weights
        compressed_macs = H * W * compressed_weights

        error = float(conv_error_for_rank(rank))
        method = "CP"
        true_rank = CONV_TRUE_RANK
        proxy = (f"{conv_toy.shape[0]}×{conv_toy.shape[1]}×"
                 f"{conv_toy.shape[2]}×{conv_toy.shape[3]}")

        extra_metric = (
            "Multiply-adds / Multiplicaciones-sumas",
            dense_macs,
            compressed_macs,
        )

    else:
        ms = ns = [8, 8, 8, 8]

        dense_weights = 4096 * 4096
        compressed_weights = tt_matrix_params(ms, ns, rank)

        error = float(tt_error_for_rank(rank))
        method = "TT"
        true_rank = TT_TRUE_RANK
        proxy = f"{tt_toy.shape[0]}×{tt_toy.shape[1]}"
        extra_metric = None

    compression = dense_weights / compressed_weights

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))

    axes[0].bar(
        ["Original", f"Compressed\n{method}"],
        [dense_weights, compressed_weights],
    )
    axes[0].set_yscale("log")
    axes[0].set_ylabel("Weights stored / Pesos guardados")
    axes[0].set_title("How much storage do we save?\n¿Cuánto almacenamiento ahorramos?")
    axes[0].grid(axis="y", alpha=0.2, which="both")

    axes[1].bar(["Error"], [error])
    axes[1].set_ylabel("Reconstruction error / Error")
    axes[1].set_title(
        f"Error on the {proxy} stand-in\nError en el sustituto {proxy}"
    )
    axes[1].grid(axis="y", alpha=0.2)

    plt.tight_layout()
    plt.show()

    print("Selected rank / Rango seleccionado:", rank)
    print()
    print(f"Original weights / Pesos originales: {dense_weights:,}")
    print(f"Compressed weights / Pesos comprimidos: {compressed_weights:,}")
    print(f"Compression / Compresión: {compression:.1f}× fewer weights / menos pesos")
    print(
        f"Reconstruction error / Error de reconstrucción: {error:.4f}  "
        f"(measured on a {proxy} stand-in of true {method} rank {true_rank}, "
        f"not on the layer above / medido sobre un sustituto, no sobre la capa)"
    )

    if extra_metric is not None:
        label, dense_macs, compressed_macs = extra_metric
        print()
        print(
            f"{label}: "
            f"{dense_macs / 1e6:.1f}M → {compressed_macs / 1e6:.1f}M"
        )

    print()
    if rank < true_rank:
        print("EN: Below the stand-in's true rank, every step down costs real accuracy.")
        print("ES: Por debajo del rango verdadero, cada paso cuesta precisión real.")
    elif rank == true_rank:
        print("EN: Exactly the stand-in's true rank — the cheapest rank that still fits it.")
        print("ES: Exactamente el rango verdadero: el más barato que aún lo representa.")
    else:
        print("EN: Past the true rank the error flattens at the noise floor: the extra")
        print("    rank buys storage cost, not accuracy. A real layer has no such")
        print("    exact rank, so its error keeps falling — slowly — instead.")
        print("ES: Pasado el rango verdadero el error se estanca en el ruido: el rango")
        print("    extra compra almacenamiento, no precisión. Una capa real no tiene")
        print("    ese rango exacto, así que su error sigue bajando, lentamente.")

    print()
    print("EN: In practice, compress a trained model and then fine-tune it.")
    print("ES: En la práctica, comprime un modelo entrenado y luego haz ajuste fino.")

compression_output = widgets.interactive_output(
    show_layer_simple,
    {"layer": layer_w, "rank": layer_rank},
)

display(widgets.VBox([
    widgets.HTML(
        "<b>Move only one control: the rank."
        "<br>Mueve un solo control: el rango.</b>"
        "<br><br>"
        "Lower rank = more compression, usually more error."
        "<br>Rango menor = más compresión, normalmente más error."
    ),
    layer_w,
    layer_rank,
    compression_output,
]))

## Exercise 2 — choose rank from a requirement / Ejercicio 2 — elige rango desde un requisito

A rank by itself is not a decision.

Instead say:

> **“I need reconstruction error ≤ threshold.”**

Then find the **smallest CP rank** that meets it.

### Steps / Pasos

1. Build the local synthetic conv kernel.
2. Evaluate CP ranks.
3. Find the first rank whose error is below the threshold.
4. Report:
   - rank;
   - reconstruction error;
   - stored weights;
   - compression ratio.

“Rank 16” is arbitrary. “The smallest rank that satisfies my error budget” is defensible, and it is the only one of the two you can put in a design document.

> 🇪🇸 Un rango por sí solo no es una decisión. Di **“necesito un error de reconstrucción ≤ umbral”** y busca el **rango CP más pequeño** que lo cumpla, informando rango, error, pesos almacenados y ratio de compresión. “Rango 16” es arbitrario; “el rango más pequeño que cumple mi presupuesto de error” es defendible.


In [ ]:
# TODO 2 / TAREA 2
#
# EN:
# 1. Test CP ranks [4, 8, 16, 24, 32, 48, 64] on conv_toy.
# 2. Compute relative reconstruction error for every rank.
# 3. Choose a threshold, for example 0.10.
# 4. Find the smallest tested rank below it.
# 5. Use rank * (3 + 3 + 512 + 512) to report the large-layer weight count.
#
# ES:
# 1. Prueba rangos CP [4, 8, 16, 24, 32, 48, 64].
# 2. Calcula el error relativo para cada rango.
# 3. Elige un umbral, por ejemplo 0.10.
# 4. Encuentra el menor rango que lo cruza.
# 5. Traduce ese rango al número de pesos de la capa grande.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

tested_ranks = [4, 8, 16, 24, 32, 48, 64]
threshold = 0.10
errors = {r: conv_error_for_rank(r) for r in tested_ranks}

eligible = [r for r in tested_ranks if errors[r] <= threshold]
best_rank = min(eligible) if eligible else None

for r in tested_ranks:
    print(f"rank {r:2d}: error={errors[r]:.4f}")

if best_rank is None:
    print("No tested rank crossed the threshold.")
else:
    large_weights = best_rank * (3 + 3 + 512 + 512)
    dense_weights = 3 * 3 * 512 * 512
    print()
    print("Smallest tested rank / Menor rango:", best_rank)
    print("Large-layer weights / Pesos:", f"{large_weights:,}")
    print("Compression / Compresión:", f"{dense_weights / large_weights:.1f}×")

## Exercise 3 — prove why TT matters / Ejercicio 3 — demuestra por qué importa TT

Keep:

- mode size `I` fixed;
- bond rank `r` fixed.

Now increase tensor order `N`.

### Compare

Dense:

`I^N`

TT:

approximately `N · I · r²`

### Your job

Show numerically and graphically that:

- dense storage grows exponentially;
- TT storage grows linearly in `N`.

> 🇪🇸 Este ejercicio no busca una factorización bonita. Busca que **veas la maldición de dimensionalidad en una gráfica**.

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. Set I=12 and r=4.
# 2. Compute TT storage for orders N=3,...,12.
# 3. Plot storage against N.
# 4. Verify np.diff(storage) is constant after the boundary terms.
# 5. Compare with dense storage I**N and Tucker r**N + N*I*r.
#
# ES:
# 1. Fija I=12 y r=4.
# 2. Calcula almacenamiento TT para N=3,...,12.
# 3. Grafícalo frente a N.
# 4. Verifica que el incremento sea constante.
# 5. Compáralo con almacenamiento denso y Tucker.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

I, r = 12, 4
orders_ex = np.arange(3, 13)
tt_store = np.array([
    2 * I * r + (N - 2) * I * r**2
    for N in orders_ex
])
dense_store = I ** orders_ex
tucker_store = r ** orders_ex + orders_ex * I * r

print("TT storage / Almacenamiento TT:", tt_store.tolist())
print("Increment per added mode / Incremento:", np.diff(tt_store).tolist())
print("Expected constant / Constante esperada:", I * r**2)
print("Linear check / Comprobación lineal:",
      np.all(np.diff(tt_store) == I * r**2))

fig, ax = plt.subplots(figsize=(8.2, 4.2))
ax.plot(orders_ex, dense_store, "o-", label="Dense")
ax.plot(orders_ex, tucker_store, "s-", label="Tucker")
ax.plot(orders_ex, tt_store, "^-", label="TT")
ax.set_yscale("log")
ax.set_xlabel("order N / orden N")
ax.set_ylabel("stored numbers / números")
ax.set_title("TT adds a local carriage; dense/Tucker cores explode")
ax.grid(alpha=0.25, which="both")
ax.legend()
plt.show()

## Step 7 — bring your own tensor / Paso 7 — trae tu propio tensor

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#16a34a,rgba(22,163,74,0))"></div>

Bring three facts: tensor shape, storage budget, and acceptable error. They determine the candidate models.

Set `(I,J,K)` and model ranks. Compare storage and the rough apply-cost proxy.

Storage and execution are separate budgets. A small representation can still be slow.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Fija la forma y los rangos. Compara almacenamiento y coste aproximado de aplicación. Son presupuestos distintos: una representación pequeña puede ser lenta.</div>

In [ ]:
#@title Budget explorer / Explorador de presupuesto — run me / ejecútame { display-mode: 'form' }

I_w = widgets.IntSlider(value=64, min=4, max=256, step=4,
                        description="I:", continuous_update=False)
J_w = widgets.IntSlider(value=64, min=4, max=256, step=4,
                        description="J:", continuous_update=False)
K_w = widgets.IntSlider(value=24, min=4, max=128, step=4,
                        description="K:", continuous_update=False)
R_w = widgets.IntSlider(value=8, min=1, max=32, step=1,
                        description="CP R:", continuous_update=False)
r1_w = widgets.IntSlider(value=4, min=1, max=16, step=1,
                         description="Tucker r1:", continuous_update=False)
r2_w = widgets.IntSlider(value=4, min=1, max=16, step=1,
                         description="Tucker r2:", continuous_update=False)
r3_w = widgets.IntSlider(value=4, min=1, max=16, step=1,
                         description="Tucker r3:", continuous_update=False)
tt_r_w = widgets.IntSlider(value=4, min=1, max=16, step=1,
                           description="TT bond r:", continuous_update=False)

def budget_table(I, J, K, R, r1, r2, r3, tt_r):
    shape = (int(I), int(J), int(K))
    ranks = (min(int(r1), shape[0]), min(int(r2), shape[1]), min(int(r3), shape[2]))
    dense = int(np.prod(shape))

    rows = [
        ("Dense", dense, 1),
        ("CP", cp_params(shape, int(R)), 3 * int(R)),
        ("Tucker", tucker_params(shape, ranks), 3 * int(np.prod(ranks))),
        ("TT", tt_params(shape, int(tt_r)), 3 * int(tt_r) ** 2),
    ]

    frame = pd.DataFrame(
        {
            "method / método": [r[0] for r in rows],
            "parameters / parámetros": [r[1] for r in rows],
            "dense / kept": [dense / r[1] for r in rows],
            "entry-cost proxy / coste por entrada": [r[2] for r in rows],
        }
    )
    display(frame.style.format({
        "parameters / parámetros": "{:,.0f}",
        "dense / kept": "{:.2f}×",
        "entry-cost proxy / coste por entrada": "{:,.0f}",
    }))

budget_out = widgets.interactive_output(
    budget_table,
    {
        "I": I_w, "J": J_w, "K": K_w, "R": R_w,
        "r1": r1_w, "r2": r2_w, "r3": r3_w, "tt_r": tt_r_w,
    },
)
display(widgets.VBox([
    widgets.HBox([I_w, J_w, K_w]),
    widgets.HBox([R_w, tt_r_w]),
    widgets.HBox([r1_w, r2_w, r3_w]),
    budget_out,
]))


## Step 8 — exact t-SVD versus truncated t-SVD / Paso 8 — t-SVD exacta frente a truncada

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#16a34a,rgba(22,163,74,0))"></div>

Exact t-SVD keeps every component. Truncated t-SVD chooses a smaller representation and accepts error.

Exact reconstruction should have error near floating-point precision. Truncation can introduce error.

Predict both errors. Then check the real taxi tensor below.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La reconstrucción exacta debe tener error cercano a la precisión numérica. Truncar puede introducir error. Predice ambos y comprueba el tensor de taxis.</div>

In [ ]:
taxi_exact = tsvd_reconstruct(T_taxi, tubal_rank=None)
taxi_rank2 = tsvd_reconstruct(T_taxi, tubal_rank=2)

print("Exact t-SVD reconstruction error / Error exacto:",
      f"{relative_error(T_taxi, taxi_exact):.3e}")
print("Tubal-rank 2 error / Error rango tubal 2:",
      f"{relative_error(T_taxi, taxi_rank2):.4f}")
print("EN: exact t-SVD is a change of representation; compression begins at truncation.")
print("ES: t-SVD exacta cambia la representación; la compresión empieza al truncar.")

## Final decision recipe / Receta final para decidir

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#16a34a,rgba(22,163,74,0))"></div>

Name every axis. State the goal. Set the budget. Measure the error. Then choose.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Nombra cada eje. Declara el objetivo. Fija el presupuesto. Mide el error. Después elige.</div>

## Where this goes next / Adónde sigue esto

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#16a34a,rgba(22,163,74,0))"></div>

Choose a topic to revisit. Posts are in English.

<details>
<summary>Further reading / Lecturas adicionales (inglés)</summary>

- [What a tensor factorization buys you](https://project-delphi.github.io/ml-blog/posts/uses-of-tensor-factorizations/)
- [Tensor factorizations and inverses](https://project-delphi.github.io/ml-blog/posts/tensor-factorizations/)
- [Tensor inverses in practice](https://project-delphi.github.io/ml-blog/posts/tensor-inverses-in-practice/)
- [Tensor inverses, worked through](https://project-delphi.github.io/ml-blog/posts/tensor-inverse-examples/)
- [Sparse tensors](https://project-delphi.github.io/ml-blog/posts/sparse_tensors/)
- [Attention as two contractions](https://project-delphi.github.io/ml-blog/posts/attention/)

</details>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige el tema que quieras repasar. Las lecturas adicionales están en inglés.</div>

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#16a34a,rgba(22,163,74,0))"></div>

## Done with this section / Fin de esta sección

Next / Siguiente: **12 · Wrap-up and take-homes / Cierre y ejercicios para casa** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/12-wrap-up-and-take-homes.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)